In [1]:
import pandas as pd
import numpy as np
import polars as pl
from pathlib import Path
import pyarrow as pa, gc
import sys
import time

import os
import warnings
import math
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupShuffleSplit



import torch
import torch.nn as nn
from torch.utils.data import Dataset

pd.set_option('display.max_columns', None) 
pd.set_option('display.max_rows', None)

warnings.filterwarnings("ignore", category=RuntimeWarning)

In [5]:
trainPath = Path("/kaggle/input/aeroclub-recsys-2025/train.parquet")
testPath = Path("/kaggle/input/aeroclub-recsys-2025/test.parquet")

# Now Create a class that process a dataframe

In [6]:
class PLDFPorcessor(): 
    def __init__(self, dfPath): 
        self.trainDFPl = pl.read_parquet(dfPath)
        self.trainDFPl = self.trainDFPl.drop([
            "Id",
            #"companyID",
            "profileId",
            "requestDate",
            #"nationality",
            "__index_level_0__",
            "miniRules1_percentage",
            "miniRules0_percentage",
            "legs1_segments3_operatingCarrier_code",
            "legs1_segments2_operatingCarrier_code",
            "legs1_segments1_operatingCarrier_code",
            "legs1_segments0_operatingCarrier_code",
            "legs0_segments3_operatingCarrier_code",
            "legs0_segments2_operatingCarrier_code",
            "legs0_segments1_operatingCarrier_code",
            "legs0_segments0_operatingCarrier_code",
            # dropping segment3 and segment2 columns as before
            "legs1_segments3_seatsAvailable",
            "legs1_segments3_flightNumber",
            "legs1_segments3_duration",
            "legs1_segments3_departureFrom_airport_iata",
            "legs1_segments3_cabinClass",
            "legs1_segments3_baggageAllowance_weightMeasurementType",
            "legs1_segments3_baggageAllowance_quantity",
            "legs1_segments3_arrivalTo_airport_iata",
            "legs1_segments3_arrivalTo_airport_city_iata",
            "legs1_segments2_seatsAvailable",
            "legs1_segments2_flightNumber",
            "legs1_segments2_duration",
            "legs1_segments2_departureFrom_airport_iata",
            "legs1_segments2_cabinClass",
            "legs1_segments2_baggageAllowance_weightMeasurementType",
            "legs1_segments2_baggageAllowance_quantity",
            "legs1_segments2_arrivalTo_airport_iata",
            "legs1_segments2_arrivalTo_airport_city_iata",
            #"legs1_segments1_seatsAvailable",
            "legs1_segments1_flightNumber",
            "legs1_segments1_duration",
            "legs1_segments1_departureFrom_airport_iata",
            #"legs1_segments1_cabinClass",
            #"legs1_segments1_baggageAllowance_weightMeasurementType",
            #"legs1_segments1_baggageAllowance_quantity",
            "legs1_segments1_arrivalTo_airport_iata",
            "legs1_segments1_arrivalTo_airport_city_iata",
            "legs0_segments3_seatsAvailable",
            "legs0_segments3_flightNumber",
            "legs0_segments3_duration",
            "legs0_segments3_departureFrom_airport_iata",
            "legs0_segments3_cabinClass",
            "legs0_segments3_baggageAllowance_weightMeasurementType",
            "legs0_segments3_baggageAllowance_quantity",
            "legs0_segments3_arrivalTo_airport_iata",
            "legs0_segments3_arrivalTo_airport_city_iata",
            "legs0_segments2_seatsAvailable",
            "legs0_segments2_flightNumber",
            "legs0_segments2_duration",
            "legs0_segments2_departureFrom_airport_iata",
            "legs0_segments2_cabinClass",
            "legs0_segments2_baggageAllowance_weightMeasurementType",
            "legs0_segments2_baggageAllowance_quantity",
            "legs0_segments2_arrivalTo_airport_iata",
            "legs0_segments2_arrivalTo_airport_city_iata",
            #"legs0_segments1_seatsAvailable",
            "legs0_segments1_flightNumber",
            "legs0_segments1_duration",
            "legs0_segments1_departureFrom_airport_iata",
            #"legs0_segments1_cabinClass",
            #"legs0_segments1_baggageAllowance_weightMeasurementType",
            #"legs0_segments1_baggageAllowance_quantity",
            "legs0_segments1_arrivalTo_airport_iata",
            "legs0_segments1_arrivalTo_airport_city_iata",
            # "corporateTariffCode",
            # These low-level iata columns dropped (if you want, keep for geo features)
            "legs0_segments0_arrivalTo_airport_city_iata", 
            "legs0_segments0_arrivalTo_airport_iata", 
            "legs0_segments0_departureFrom_airport_iata", 
            "legs0_segments0_duration", 
            "legs0_segments0_flightNumber", 
            "legs1_segments0_arrivalTo_airport_city_iata", 
            "legs1_segments0_arrivalTo_airport_iata", 
            "legs1_segments0_departureFrom_airport_iata", 
            "legs1_segments0_duration", 
            "legs1_segments0_flightNumber",
        ])
        pa.default_memory_pool().release_unused()
        gc.collect()

    
    def add_price_rank_features(self):
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("totalPrice").rank("dense").over("ranker_id").alias("price_rank"),
            pl.len().over("ranker_id").alias("group_count"),
            pl.col("legs0_duration").rank("dense").over("ranker_id").alias("duration_rank"),
        ])
        self.trainDFPl = self.trainDFPl.with_columns([
            ((pl.col("price_rank") - 1) / (pl.col("group_count") - 1).cast(pl.Float64)).alias("price_pct_rank")
        ])
        self.trainDFPl = self.trainDFPl.drop("group_count")
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") + 1).log().alias("log_price")
        ])
        print("Done add_price_rank_features!!!")
        

    def add_min_stopage_per_trip(self): 
        # Reusing your logic, renamed to snake_case to be consistent
        freq_split = self.trainDFPl['frequentFlyer'].fill_null("").str.split("/")
        codeCols = [
            "legs0_segments0_marketingCarrier_code",
            "legs0_segments1_marketingCarrier_code",
            "legs0_segments2_marketingCarrier_code",
            "legs0_segments3_marketingCarrier_code",
            "legs1_segments0_marketingCarrier_code",
            "legs1_segments1_marketingCarrier_code",
            "legs1_segments2_marketingCarrier_code",
            "legs1_segments3_marketingCarrier_code",
        ]
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.sum_horizontal(pl.col(col).is_not_null().cast(pl.UInt8) for col in codeCols).alias("seg_tempor"), 
            pl.col("corporateTariffCode").is_not_null().cast(pl.Int32).alias("has_corporate_tariff")
        ])
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("seg_tempor") == pl.col("seg_tempor").min().over("ranker_id")).cast(pl.Int32).alias("minStopagePerRankderID")
        ])
        self.trainDFPl = self.trainDFPl.drop("seg_tempor")

        self.trainDFPl = self.trainDFPl.with_columns([
            ((pl.col("miniRules1_monetaryAmount") == 0) & (pl.col("miniRules1_statusInfos") == 1)).cast(pl.Int8).alias("free_exchange")
        ])

        self.trainDFPl = self.trainDFPl.with_columns([
            freq_split.alias("freq_split"),  
            pl.sum_horizontal([pl.col(col).is_in(freq_split).cast(pl.Int32)for col in codeCols]).alias("frequentFlyer_marketingCarrier_match")
        ])
        self.trainDFPl = self.trainDFPl.drop("freq_split")

        del codeCols, freq_split
        pa.default_memory_pool().release_unused()
        gc.collect()
        
        print("Done add_min_stopage_per_trip!!!")


    

    def ff_flyer_bin_converter(self): 
        freq_split = self.trainDFPl['frequentFlyer'].fill_null("").str.split("/")

        codeCols = [
            "legs0_segments0_marketingCarrier_code",
            "legs0_segments1_marketingCarrier_code",
            "legs0_segments2_marketingCarrier_code",
            "legs0_segments3_marketingCarrier_code",
            "legs1_segments0_marketingCarrier_code",
            "legs1_segments1_marketingCarrier_code",
            "legs1_segments2_marketingCarrier_code",
            "legs1_segments3_marketingCarrier_code",
        ]
        
        self.trainDFPl = self.trainDFPl.with_columns(freq_split.alias("freq_split"))
        
        matches_exprs = [
            pl.when(pl.col(col).is_in(pl.col("freq_split"))).then(1).otherwise(0).alias(f"match_{col}")
            for col in codeCols
        ]
        
        self.trainDFPl = self.trainDFPl.with_columns(matches_exprs)
        
        match_cols = [f"match_{col}" for col in codeCols]
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.sum_horizontal(match_cols).alias("numberof_same_frequentFlyter_operator"),
            pl.col("bySelf").cast(pl.Int8).alias("bySelf")
        )
        
        drop_cols = codeCols + ["frequentFlyer"] + match_cols + ["freq_split"]
        self.trainDFPl = self.trainDFPl.drop(drop_cols)
        
        del codeCols, freq_split, matches_exprs, match_cols, drop_cols
        pa.default_memory_pool().release_unused()
        gc.collect()

        # Convert binary features to int8 safely
        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("isAccess3D").fill_null(False).cast(pl.Int8),
            pl.col("isVip").fill_null(False).cast(pl.Int8),
            pl.col("sex").fill_null(False).cast(pl.Int8),
            pl.col("has_corporate_tariff").fill_null(False).cast(pl.Int8),
        ])
        
        # Aircraft code presence binary
        aircraft_cols = [
            "legs0_segments0_aircraft_code",
            "legs0_segments1_aircraft_code",
            "legs0_segments2_aircraft_code",
            "legs0_segments3_aircraft_code",
            "legs1_segments0_aircraft_code",
            "legs1_segments1_aircraft_code",
            "legs1_segments2_aircraft_code",
            "legs1_segments3_aircraft_code",
        ]
        self.trainDFPl = self.trainDFPl.with_columns(
            [pl.col(c).is_not_null().cast(pl.Int8).alias(c) for c in aircraft_cols]
        )
        
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.sum_horizontal(aircraft_cols).alias("total_travel_stopage")
        )
        self.trainDFPl = self.trainDFPl.with_columns(
            pl.col("total_travel_stopage").min().over("ranker_id").alias("minimum_travel_segment"),
        )
        self.trainDFPl = self.trainDFPl.drop(aircraft_cols)
        pa.default_memory_pool().release_unused()
        gc.collect()
        print("Done ff_flyer_bin_converter!")


    def hhmmss_to_minutes(self, col_name) -> pl.Expr:
        def parse_duration(x):
            if len(x) < 3:
                return None
            first_part = x[0]
            minutes = float(x[1])
            seconds = float(x[2]) if len(x) > 2 else 0
            if '.' in first_part:
                day_str, hour_str = first_part.split('.')
                days = int(day_str)
                hours = int(hour_str)
            else:
                days = 0
                hours = int(first_part)
            total_minutes = days * 24 * 60 + hours * 60 + minutes + seconds / 60
            return total_minutes
    
        return (
            pl.col(col_name)
            .fill_null("00:00:00")
            .str.split(":")
            .map_elements(parse_duration, return_dtype=pl.Float64)
            .alias(col_name)
        )
    
    def hour_min_and_tax_converter(self): 
        self.trainDFPl = self.trainDFPl.with_columns([
            self.hhmmss_to_minutes("legs0_duration"),
            self.hhmmss_to_minutes("legs1_duration"),
        ])
        
        self.trainDFPl = self.trainDFPl.with_columns(
            (pl.col("legs0_duration") + pl.col("legs1_duration")).alias("total_travel_time_in_minutes")
        )
    
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("searchRoute").str.len_chars() // 3).alias("searchRoute"),
            (pl.col("taxes") / pl.col("totalPrice")).alias("tax_percentage"),
        ])

        self.trainDFPl = self.trainDFPl.with_columns([
            pl.col("legs0_duration").rank("dense").over("ranker_id").alias("legs0_duration_rank"),
            pl.col("legs1_duration").rank("dense").over("ranker_id").alias("legs1_duration_rank"),
            pl.col("total_travel_time_in_minutes").rank("dense").over("ranker_id").alias("total_travel_time_in_minutes_duration_rank"),
            
            #pl.col("total_travel_time_in_minutes").max().over("ranker_id").alias("total_travel_time_in_minutes_duration_rank_max"),
            #pl.col("total_travel_time_in_minutes").mean().over("ranker_id").alias("total_travel_time_in_minutes_duration_rank_mean"),
            #pl.col("total_travel_time_in_minutes").std().over("ranker_id").alias("total_travel_time_in_minutes_duration_rank_std"),
        ])
        gc.collect()
        print("Done hour_min_and_tax_converter")

    def others_and_duration(self):
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("taxes") / pl.col("pricingInfo_passengerCount")).alias("taxes"),
            (pl.col("totalPrice")/pl.col("pricingInfo_passengerCount")).alias("totalPrice")
        ])

        min_vals = (
            self.trainDFPl
            .group_by("ranker_id") 
            .agg([
                pl.col("totalPrice").min().alias("min_totalPrice"),
                pl.col("legs0_duration").min().alias("min_legs0_duration"),
                pl.col("legs1_duration").min().alias("min_legs1_duration"),
                pl.col("total_travel_time_in_minutes").min().alias("min_total_travel_time"),
            ])
        )
        
        self.trainDFPl = self.trainDFPl.join(min_vals, on="ranker_id", how="left")
        
        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("totalPrice") - pl.col("min_totalPrice")).alias("totalPrice_diff"),
            (pl.col("legs0_duration") - pl.col("min_legs0_duration")).alias("legs0Duration_diff"),
            (pl.col("legs1_duration") - pl.col("min_legs1_duration")).alias("legs1Duration_diff"),
            (pl.col("total_travel_time_in_minutes") - pl.col("min_total_travel_time")).alias("totalDuration_diff"),
        ])
        
        self.trainDFPl = self.trainDFPl.drop([
            "min_totalPrice",
            "min_legs0_duration",
            "min_legs1_duration",
            "min_total_travel_time", 
            "pricingInfo_passengerCount",
            "corporateTariffCode"
        ])

        self.trainDFPl = self.trainDFPl.with_columns(
            ((pl.col("totalPrice") + 1)/(pl.col("legs0_duration").fill_null(0) + pl.col("legs1_duration").fill_null(0) + 1)).alias("priceDuration_TradeOff")
        )        
        gc.collect()
        print("Done others_and_duration")

    def arrival_and_monetory_adder(self): 
        cols = ['legs0_arrivalAt', 'legs0_departureAt', 'legs1_arrivalAt', 'legs1_departureAt']
        for x in cols:
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col(x).str.strip_chars().str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M:%S", strict=False).alias(f"{x}_parsed")
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                pl.col(f"{x}_parsed").dt.hour().fill_null(0).alias(f"{x}_hour"),
                pl.col(f"{x}_parsed").dt.minute().fill_null(0).alias(f"{x}_minute"),
                pl.col(f"{x}_parsed").dt.weekday().fill_null(0).alias(f"{x}_weekday"),  # added weekday here
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col(f"{x}_hour") + (pl.col(f"{x}_minute") / 60)).alias(f"{x}_decimal_hour")
            ])
        
            self.trainDFPl = self.trainDFPl.with_columns([
                (((pl.col(f"{x}_hour") * 3600 + pl.col(f"{x}_minute") * 60) / 86400) * 360).alias(f"{x}_angle_deg")
            ])

            # Add cyclical encoding for weekday (0=Mon, 6=Sun)
            self.trainDFPl = self.trainDFPl.with_columns([
                (2 * np.pi * pl.col(f"{x}_weekday") / 7).map_elements(np.sin, return_dtype=pl.Float64).alias(f"{x}_weekday_sin"),
                (2 * np.pi * pl.col(f"{x}_weekday") / 7).map_elements(np.cos, return_dtype=pl.Float64).alias(f"{x}_weekday_cos"),
            ])
            
            #  Red-eye indicator (late night/early morning)
            self.trainDFPl = self.trainDFPl.with_columns([
                ((self.trainDFPl[f"{x}_hour"] >= 23) | (self.trainDFPl[f"{x}_hour"] < 6)).cast(pl.Int32).alias(f"{x}_is_redeye")
            ])


            # Drop intermediates
            self.trainDFPl = self.trainDFPl.drop([f"{x}_parsed", f"{x}_hour", f"{x}_minute"])

        del cols
        gc.collect()

        self.trainDFPl = self.trainDFPl.with_columns([
            (pl.col("miniRules0_monetaryAmount") / pl.col("totalPrice")).alias("miniRules0_monetaryAmount_ratio"),
            (pl.col("miniRules1_monetaryAmount") / pl.col("totalPrice")).alias("miniRules1_monetaryAmount_ratio"),
        ])
        
        self.trainDFPl = self.trainDFPl.drop([
            "miniRules0_monetaryAmount",
            "miniRules1_monetaryAmount",
            "legs0_arrivalAt",
            "legs0_departureAt",
            "legs1_arrivalAt",
            "legs1_departureAt",
            "legs0_duration",
            "legs1_duration",
            "bySelf",
        ])
        gc.collect()
        print("Done arrival_and_monetory_adder")


    def add_interaction_features(self):
        # Example: interaction between isVip and free_exchange
        if "isVip" in self.trainDFPl.columns and "free_exchange" in self.trainDFPl.columns:
            self.trainDFPl = self.trainDFPl.with_columns([
                (pl.col("isVip") * pl.col("free_exchange")).alias("vip_free_exchange_interaction")
            ])
        gc.collect()
        print("Done add_interaction_features")

    def returnProcessedDF(self):
        self.add_price_rank_features()
        self.add_min_stopage_per_trip()
        self.ff_flyer_bin_converter()
        self.hour_min_and_tax_converter()
        self.others_and_duration()
        self.arrival_and_monetory_adder()
        self.add_interaction_features()
        return self.trainDFPl


In [7]:
trainDFPl = PLDFPorcessor(trainPath).returnProcessedDF() #.sort("ranker_id")
trainDFPl.head()

Done add_price_rank_features!!!
Done add_min_stopage_per_trip!!!
Done ff_flyer_bin_converter!
Done hour_min_and_tax_converter
Done others_and_duration
Done arrival_and_monetory_adder
Done add_interaction_features


companyID,nationality,isAccess3D,isVip,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_seatsAvailable,legs0_segments1_baggageAllowance_quantity,legs0_segments1_baggageAllowance_weightMeasurementType,legs0_segments1_cabinClass,legs0_segments1_seatsAvailable,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_seatsAvailable,legs1_segments1_baggageAllowance_quantity,legs1_segments1_baggageAllowance_weightMeasurementType,legs1_segments1_cabinClass,legs1_segments1_seatsAvailable,miniRules0_statusInfos,miniRules1_statusInfos,pricingInfo_isAccessTP,ranker_id,searchRoute,sex,taxes,totalPrice,selected,price_rank,duration_rank,price_pct_rank,log_price,has_corporate_tariff,minStopagePerRankderID,free_exchange,frequentFlyer_marketingCarrier_match,…,total_travel_time_in_minutes,tax_percentage,legs0_duration_rank,legs1_duration_rank,total_travel_time_in_minutes_duration_rank,totalPrice_diff,legs0Duration_diff,legs1Duration_diff,totalDuration_diff,priceDuration_TradeOff,legs0_arrivalAt_weekday,legs0_arrivalAt_decimal_hour,legs0_arrivalAt_angle_deg,legs0_arrivalAt_weekday_sin,legs0_arrivalAt_weekday_cos,legs0_arrivalAt_is_redeye,legs0_departureAt_weekday,legs0_departureAt_decimal_hour,legs0_departureAt_angle_deg,legs0_departureAt_weekday_sin,legs0_departureAt_weekday_cos,legs0_departureAt_is_redeye,legs1_arrivalAt_weekday,legs1_arrivalAt_decimal_hour,legs1_arrivalAt_angle_deg,legs1_arrivalAt_weekday_sin,legs1_arrivalAt_weekday_cos,legs1_arrivalAt_is_redeye,legs1_departureAt_weekday,legs1_departureAt_decimal_hour,legs1_departureAt_angle_deg,legs1_departureAt_weekday_sin,legs1_departureAt_weekday_cos,legs1_departureAt_is_redeye,miniRules0_monetaryAmount_ratio,miniRules1_monetaryAmount_ratio,vip_free_exchange_interaction
i64,i64,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u32,i8,f64,f64,i64,u32,u32,f64,f64,i8,i32,i8,i32,…,f64,f64,u32,u32,u32,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,i32,i8,f64,f64,f64,f64,i32,i8,f64,f64,f64,f64,i32,i8,f64,f64,f64,f64,i32,f64,f64,i8
57323,36,0,0,1.0,0.0,1.0,9.0,null,null,null,null,1.0,0.0,1.0,9.0,null,null,null,null,null,null,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,370.0,16884.0,1,1,1,0.0,9.734181,0,1,null,0,…,315.0,0.021914,1,1,1,0.0,0.0,0.0,0.0,53.433544,6,16.333333,-33.4,-0.781831,0.62349,0,6,15.666667,-47.666667,-0.781831,0.62349,0,2,14.333333,-63.4,0.974928,-0.222521,0,2,9.75,134.516667,0.974928,-0.222521,0,null,null,null
57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,51125.0,0,2,2,0.041667,10.842048,1,0,0,4,…,950.0,0.043814,2,2,2,34241.0,285.0,350.0,635.0,53.760252,6,14.833333,-63.366667,-0.781831,0.62349,0,6,9.416667,134.85,-0.781831,0.62349,0,3,8.5,120.033333,0.433884,-0.900969,0,2,22.083333,57.116667,0.974928,-0.222521,0,0.044988,0.06846,0
57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,53695.0,0,3,2,0.083333,10.891094,0,0,0,4,…,950.0,0.041717,2,2,2,36811.0,285.0,350.0,635.0,56.462671,6,14.833333,-63.366667,-0.781831,0.62349,0,6,9.416667,134.85,-0.781831,0.62349,0,3,8.5,120.033333,0.433884,-0.900969,0,2,22.083333,57.116667,0.974928,-0.222521,0,0.042835,0.065183,0
57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,81880.0,0,4,2,0.125,11.313022,1,0,1,4,…,950.0,0.027357,2,2,2,64996.0,285.0,350.0,635.0,86.099895,6,14.833333,-63.366667,-0.781831,0.62349,0,6,9.416667,134.85,-0.781831,0.62349,0,3,8.5,120.033333,0.433884,-0.900969,0,2,22.083333,57.116667,0.974928,-0.222521,0,0.0,0.0,0
57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,86070.0,0,5,2,0.16

## Now handle the Nan values cause Tensor can not hangle Nan values by default

In [8]:
trainDFPl = trainDFPl.with_columns(
    pl.col("legs0_segments0_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
    pl.col("legs0_segments0_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
    pl.col("legs0_segments0_seatsAvailable").fill_null(-1) , #.fill_nan(0)
    
    pl.col("legs0_segments1_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
    pl.col("legs0_segments1_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
    pl.col("legs0_segments1_seatsAvailable").fill_null(-1) , #.fill_nan(0)
    pl.col("legs0_segments1_cabinClass").fill_null(-1) , #.fill_nan(0)
    
    pl.col("legs1_segments0_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
    pl.col("legs1_segments0_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
    pl.col("legs1_segments0_seatsAvailable").fill_null(-1) , #.fill_nan(0)
    pl.col("legs1_segments0_cabinClass").fill_null(-1) ,
    
    pl.col("legs1_segments1_baggageAllowance_quantity").fill_null(-1) , #.fill_nan(0)
    pl.col("legs1_segments1_baggageAllowance_weightMeasurementType").fill_null(-1) , #.fill_nan(0)
    pl.col("legs1_segments1_seatsAvailable").fill_null(-1) , #.fill_nan(0)
    pl.col("legs1_segments1_cabinClass").fill_null(-1) ,
    
    pl.col("miniRules0_statusInfos").fill_null(-1) ,
    pl.col("miniRules1_statusInfos").fill_null(-1) ,
    pl.col("pricingInfo_isAccessTP").fill_null(-1) ,
    pl.col("free_exchange").fill_null(-1) ,
    pl.col("miniRules0_monetaryAmount_ratio").fill_null(-1) ,
    pl.col("miniRules1_monetaryAmount_ratio").fill_null(-1) ,
    pl.col("vip_free_exchange_interaction").fill_null(-1) ,
)

trainDFPl.head(15)

companyID,nationality,isAccess3D,isVip,legs0_segments0_baggageAllowance_quantity,legs0_segments0_baggageAllowance_weightMeasurementType,legs0_segments0_cabinClass,legs0_segments0_seatsAvailable,legs0_segments1_baggageAllowance_quantity,legs0_segments1_baggageAllowance_weightMeasurementType,legs0_segments1_cabinClass,legs0_segments1_seatsAvailable,legs1_segments0_baggageAllowance_quantity,legs1_segments0_baggageAllowance_weightMeasurementType,legs1_segments0_cabinClass,legs1_segments0_seatsAvailable,legs1_segments1_baggageAllowance_quantity,legs1_segments1_baggageAllowance_weightMeasurementType,legs1_segments1_cabinClass,legs1_segments1_seatsAvailable,miniRules0_statusInfos,miniRules1_statusInfos,pricingInfo_isAccessTP,ranker_id,searchRoute,sex,taxes,totalPrice,selected,price_rank,duration_rank,price_pct_rank,log_price,has_corporate_tariff,minStopagePerRankderID,free_exchange,frequentFlyer_marketingCarrier_match,…,total_travel_time_in_minutes,tax_percentage,legs0_duration_rank,legs1_duration_rank,total_travel_time_in_minutes_duration_rank,totalPrice_diff,legs0Duration_diff,legs1Duration_diff,totalDuration_diff,priceDuration_TradeOff,legs0_arrivalAt_weekday,legs0_arrivalAt_decimal_hour,legs0_arrivalAt_angle_deg,legs0_arrivalAt_weekday_sin,legs0_arrivalAt_weekday_cos,legs0_arrivalAt_is_redeye,legs0_departureAt_weekday,legs0_departureAt_decimal_hour,legs0_departureAt_angle_deg,legs0_departureAt_weekday_sin,legs0_departureAt_weekday_cos,legs0_departureAt_is_redeye,legs1_arrivalAt_weekday,legs1_arrivalAt_decimal_hour,legs1_arrivalAt_angle_deg,legs1_arrivalAt_weekday_sin,legs1_arrivalAt_weekday_cos,legs1_arrivalAt_is_redeye,legs1_departureAt_weekday,legs1_departureAt_decimal_hour,legs1_departureAt_angle_deg,legs1_departureAt_weekday_sin,legs1_departureAt_weekday_cos,legs1_departureAt_is_redeye,miniRules0_monetaryAmount_ratio,miniRules1_monetaryAmount_ratio,vip_free_exchange_interaction
i64,i64,i8,i8,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,u32,i8,f64,f64,i64,u32,u32,f64,f64,i8,i32,i8,i32,…,f64,f64,u32,u32,u32,f64,f64,f64,f64,f64,i8,f64,f64,f64,f64,i32,i8,f64,f64,f64,f64,i32,i8,f64,f64,f64,f64,i32,i8,f64,f64,f64,f64,i32,f64,f64,i8
57323,36,0,0,1.0,0.0,1.0,9.0,-1.0,-1.0,-1.0,-1.0,1.0,0.0,1.0,9.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,370.0,16884.0,1,1,1,0.0,9.734181,0,1,-1,0,…,315.0,0.021914,1,1,1,0.0,0.0,0.0,0.0,53.433544,6,16.333333,-33.4,-0.781831,0.62349,0,6,15.666667,-47.666667,-0.781831,0.62349,0,2,14.333333,-63.4,0.974928,-0.222521,0,2,9.75,134.516667,0.974928,-0.222521,0,-1.0,-1.0,-1
57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,51125.0,0,2,2,0.041667,10.842048,1,0,0,4,…,950.0,0.043814,2,2,2,34241.0,285.0,350.0,635.0,53.760252,6,14.833333,-63.366667,-0.781831,0.62349,0,6,9.416667,134.85,-0.781831,0.62349,0,3,8.5,120.033333,0.433884,-0.900969,0,2,22.083333,57.116667,0.974928,-0.222521,0,0.044988,0.06846,0
57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,53695.0,0,3,2,0.083333,10.891094,0,0,0,4,…,950.0,0.041717,2,2,2,36811.0,285.0,350.0,635.0,56.462671,6,14.833333,-63.366667,-0.781831,0.62349,0,6,9.416667,134.85,-0.781831,0.62349,0,3,8.5,120.033333,0.433884,-0.900969,0,2,22.083333,57.116667,0.974928,-0.222521,0,0.042835,0.065183,0
57323,36,1,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,81880.0,0,4,2,0.125,11.313022,1,0,1,4,…,950.0,0.027357,2,2,2,64996.0,285.0,350.0,635.0,86.099895,6,14.833333,-63.366667,-0.781831,0.62349,0,6,9.416667,134.85,-0.781831,0.62349,0,3,8.5,120.033333,0.433884,-0.900969,0,2,22.083333,57.116667,0.974928,-0.222521,0,0.0,0.0,0
57323,36,0,0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,4.0,1.0,0.0,1.0,9.0,1.0,0.0,1.0,9.0,1.0,1.0,1.0,"""98ce0dabf6964640b63079fbafd42c…",4,1,2240.0,86070.0,0,5,2,0.166667

In [9]:
for col_name in trainDFPl.columns:
    total_missing = (trainDFPl[col_name].is_null().sum()/len(trainDFPl[col_name]))*100
    print(f"{col_name}: {total_missing}")

gc.collect()

companyID: 0.0
nationality: 0.0
isAccess3D: 0.0
isVip: 0.0
legs0_segments0_baggageAllowance_quantity: 0.0
legs0_segments0_baggageAllowance_weightMeasurementType: 0.0
legs0_segments0_cabinClass: 0.0
legs0_segments0_seatsAvailable: 0.0
legs0_segments1_baggageAllowance_quantity: 0.0
legs0_segments1_baggageAllowance_weightMeasurementType: 0.0
legs0_segments1_cabinClass: 0.0
legs0_segments1_seatsAvailable: 0.0
legs1_segments0_baggageAllowance_quantity: 0.0
legs1_segments0_baggageAllowance_weightMeasurementType: 0.0
legs1_segments0_cabinClass: 0.0
legs1_segments0_seatsAvailable: 0.0
legs1_segments1_baggageAllowance_quantity: 0.0
legs1_segments1_baggageAllowance_weightMeasurementType: 0.0
legs1_segments1_cabinClass: 0.0
legs1_segments1_seatsAvailable: 0.0
miniRules0_statusInfos: 0.0
miniRules1_statusInfos: 0.0
pricingInfo_isAccessTP: 0.0
ranker_id: 0.0
searchRoute: 0.0
sex: 0.0
taxes: 0.0
totalPrice: 0.0
selected: 0.0
price_rank: 0.0
duration_rank: 0.0
price_pct_rank: 0.0
log_price: 0.0
has_c

0

## ======================== Now split the dataset in train and test ==========================

In [10]:
ranker_ids = trainDFPl["ranker_id"].unique().to_numpy()

# split DF based on ranker_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.01, random_state=42)
train_idx, valid_idx = next(gss.split(ranker_ids, groups=ranker_ids))

train_rankers = ranker_ids[train_idx]
valid_rankers = ranker_ids[valid_idx] # 1056 

print(len(train_rankers), len(valid_rankers))

train_df = trainDFPl.filter(pl.col("ranker_id").is_in(train_rankers))
valid_df = trainDFPl.filter(pl.col("ranker_id").is_in(valid_rankers))

104483 1056


In [11]:
# gss, train_idx, valid_idx , trainDFPl, train_rankers, valid_rankers = None, None, None, None, None, None
del gss, train_idx, valid_idx , trainDFPl
pa.default_memory_pool().release_unused()
gc.collect()

0

# Now group them and make tensor from them 

In [14]:
class FlightNumericalEncoder:
    def __init__(self, feature_cols):
        self.feature_cols = feature_cols

    def encode_group(self, group_df):
        selected_col = group_df["selected"].to_list()
        features_df = group_df[self.feature_cols]

        # Convert to tensor, handling NaN values
        features_array = features_df.to_numpy().astype(np.float32)
        features_tensor = torch.from_numpy(features_array)

        # Create target tensor
        target = torch.zeros(len(features_df))
        target[selected_col.index(1)] = 1

        return features_tensor, target # features_tensor.shape=(number of row/flight in group, number of col=75)  ||| target.shape=([number of row/flight in group])

gc.collect()

7

## ======================= Now Build the core model ==============================

In [15]:
class NumericalFlightRanker(nn.Module):
    def __init__(self, num_features=75, dropout=0.1, hidden_dim=256, lstm_hidden=128, cnn_channels=64): #  num_features = number of col
        super().__init__()
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.dropout = nn.Dropout(dropout)
        
        # Feature preprocessing layers
        self.feature_embedding = nn.Sequential(
            nn.Linear(num_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # define the core hidden dim part : =================================
        self.dropout = nn.Dropout(dropout)
        
        # CNN layers to capture patterns at different scales
        self.cnn = nn.Sequential(
            nn.Conv1d(in_channels=hidden_dim, out_channels=cnn_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(cnn_channels, cnn_channels, kernel_size=7, padding=3),
            nn.ReLU()
        )
        # Bidirectional LSTM to capture sequence-level dependencies
        self.lstm = nn.LSTM(
            input_size=cnn_channels,
            hidden_size=lstm_hidden,
            num_layers=2,
            batch_first=True,
            bidirectional=True
        )
        # Self-attention
        self.attention = nn.Sequential(
            nn.Linear(lstm_hidden*2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )
        # Fully connected layers for final ranking score
        self.fc = nn.Sequential(
            nn.Linear(lstm_hidden*2, 128),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

        


    def forward(self, features):
        # features: (num_flights/row, num_features) = [35 flight . 75 col/feature]
        
        # Embed features
        embedded_features = self.feature_embedding(features)         # (num_flights, hidden_dim) = [35, 256]
        embedded_features = embedded_features.unsqueeze(0)           # (1, num_flights, hidden_dim)= [1, 35, 256]
        
        # Get ranking scores
        embedded_features = embedded_features.transpose(1, 2)        # (batch_size=1, hidden_dim, num_flights) for CNN                     = [1, 256, 35]
        embedded_features = self.cnn(embedded_features)              # (batch_size=1, cnn_channels , num_flights)                          =  [1, 64, 35]                         
        embedded_features = embedded_features.transpose(1, 2)        # (batch_size, num_flights, cnn_channels) for LSTM                    =  [1, 35, 64] 

        lstm_out, _ = self.lstm(embedded_features)                   # (batch_size, num_flights, lstm_hidden*2)                            = [1, 35, 128*2]
        
        # Self-attention gating per sentence
        gates = self.attention(lstm_out)                             # (batch_size, num_flights, 1)                                         = [1, 35, 1]
        lstm_out = lstm_out * gates                                  # apply gate (element-wise). same shape as LSTM but different weight   =  [1, 35, 128*2] 

        
        scores = self.fc(lstm_out).squeeze(-1)                       # (batch_size, seq_len)                                                = [1, 35]
        scores = scores.squeeze(0)                                   # (num_flights,)
        return scores

gc.collect()

8

## ================ Now define some Utility function: "pairwise_ranking_loss" and "eval_hit_k" ===================

In [34]:
import torch.nn.functional as F

def listwise_ranking_loss(scores, labels):
    """
    Alternative listwise loss using ranking-based approach.
    This version considers all pairwise relationships but in a more efficient way.
    
    Args:
        scores: Model predictions for each item [batch_size]
        labels: Relevance labels for each item [batch_size]
    
    Returns:
        loss: Computed listwise loss
    """
    if len(scores) <= 1:
        return torch.tensor(0.0, device=scores.device, requires_grad=True)
    
    # Create all pairwise differences
    score_diff = scores.unsqueeze(0) - scores.unsqueeze(1)  # [n, n]
    label_diff = labels.unsqueeze(0) - labels.unsqueeze(1)  # [n, n]
    
    # Only consider pairs where one item is more relevant than another
    valid_pairs = (label_diff > 0).float()
    
    if valid_pairs.sum() == 0:
        return torch.tensor(0.0, device=scores.device, requires_grad=True)
    
    # Use sigmoid loss for ranking
    ranking_loss = F.logsigmoid(score_diff) * valid_pairs
    loss = -ranking_loss.sum() / valid_pairs.sum()
    
    return loss

In [18]:
class Eval_Hit_3(): 
    def __init__(self, eval_def, rowEncoder, device): 
        self.eval_def = eval_def
        self.rowEncoder = rowEncoder
        self.device = device

    def prepare_for_validation(self): 
        self.eval_ranker_embeddings = []
        invalid_rankers = self.eval_def["ranker_id"].unique()
        
        for evalRankerIds in invalid_rankers:
            eval_filtered_df = self.eval_def.filter(
                pl.col("ranker_id") == evalRankerIds
            )
            flight_texts, eval_target = self.rowEncoder.encode_group(eval_filtered_df)
            self.eval_ranker_embeddings.append({
                "flight_texts": flight_texts.to(self.device),
                "eval_target": eval_target.to(self.device)
            })
        gc.collect()

    
    def getHit_3(self, trainedModel):
        trainedModel.eval()
        
        total_hits = 0
        total_samples = 0
        skipped = 0
        
        for indx, infos in enumerate(self.eval_ranker_embeddings):
            flight_texts = infos["flight_texts"]
            eval_target = infos["eval_target"]
            
            scores = trainedModel(flight_texts)
            if len(scores) < 3:
                skipped +=1
                continue
            
            top_k_indices = torch.topk(scores, 3).indices
            selected_idx = torch.argmax(eval_target)

            # Check if selected flight is in top-k
            if selected_idx in top_k_indices:
                total_hits += 1
            total_samples += 1

            #del flight_texts, eval_target, scores, top_k_indices, selected_idx
            #torch.cuda.empty_cache() if torch.cuda.is_available() else None
            gc.collect()

        hit_at_k = total_hits / total_samples
        print("Number of skipped eval ranker-id: ", skipped)
        return hit_at_k

gc.collect()

0

# ============= Now start training the model ===================

In [19]:
model_path = "/kaggle/working/minilm_ranker.pt"
optimizer_path = "/kaggle/working/optimizer.pt"

In [25]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
feature_cols = [col for col in train_df.columns if col not in ['ranker_id', 'selected', "Id"]]


model = NumericalFlightRanker(
    num_features=len(feature_cols), 
    dropout=0.1, 
    hidden_dim=256, 
    lstm_hidden=128, 
    cnn_channels=64
).to(device)

flight_encoder = FlightNumericalEncoder(feature_cols)
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=0.0005,
    weight_decay=0.01 
)

# Add learning rate scheduler
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, verbose=True
)

# Load weights only if files exist
if os.path.exists(model_path) and os.path.exists(optimizer_path):
    model.load_state_dict(torch.load(model_path, map_location=device))
    optimizer.load_state_dict(torch.load(optimizer_path, map_location=device))
    print("Loaded saved model and optimizer weights.")


hitrateEval_3 = Eval_Hit_3(valid_df, flight_encoder, device)
hitrateEval_3.prepare_for_validation()
gc.collect()

0

In [35]:
trainUniqueRanker = train_df["ranker_id"].unique()
best_hitrate = 0.0


for epoch in range(3):
    model.train()
    total_loss_per_epoch = 0

    lossTempStor = []
    
    for indx, ranker_id in enumerate(trainUniqueRanker):
        model.train()
        filtered_df = train_df.filter(
            pl.col("ranker_id") == ranker_id
        )
        
        features_tensor, target = flight_encoder.encode_group(filtered_df)
        features_tensor = features_tensor.to(device)
        target = target.to(device)

        optimizer.zero_grad()
        scores = model(features_tensor)
        loss = listwise_ranking_loss(scores, target)

        loss.backward()
        optimizer.step()
    
        total_loss_per_epoch += loss.item()
        lossTempStor.append(f"{loss.item():.4f}")

        if indx % 10 == 0 and indx!=0: 
            print(f"Epoch:{epoch} || ranker Index:{indx} || Loss:{lossTempStor} || Avg Loss:{total_loss_per_epoch/(indx + 1):.4f}")
            lossTempStor = []

            # Update learning rate
            avg_epoch_loss = total_loss_per_epoch /(indx+1)
            scheduler.step(avg_epoch_loss)

        # save the trained model at each 1000th ranker id
        if (indx%500 == 0 and indx!=0) or ((indx-1)==len(train_rankers)): 
            # Remove old files if they exist
            if os.path.exists(model_path):
                os.remove(model_path)
            if os.path.exists(optimizer_path):
                os.remove(optimizer_path)
            
            torch.save(model.state_dict(), model_path)
            torch.save(optimizer.state_dict(), optimizer_path)
            print("Model and optimizer saved successfully!")

            hitrateEval3 = hitrateEval_3.getHit_3(model)
            print(f"hitrate@3: {hitrateEval3:.5f}")
            print()
    
        del features_tensor, target, scores
        #torch.cuda.empty_cache()
        gc.collect()

    torch.cuda.empty_cache() if torch.cuda.is_available() else None
    gc.collect()
    print(f"Loss in Epoch:{epoch} is -> {total_loss_per_epoch/len(trainUniqueRanker):.4f}")
    

Epoch:0 || ranker Index:10 || Loss:['0.8243', '0.8544', '0.6422', '0.7811', '0.5713', '1.1189', '0.6389', '0.8951', '0.8564', '0.7209', '0.6813'] || Avg Loss:0.7804
Epoch:0 || ranker Index:20 || Loss:['0.8166', '0.5479', '0.6498', '0.7828', '0.6263', '0.7869', '0.8061', '0.7442', '0.7219', '0.8075'] || Avg Loss:0.7559
Epoch:0 || ranker Index:30 || Loss:['0.7056', '0.8674', '0.6666', '0.9559', '0.7441', '0.7174', '0.7119', '0.7218', '0.6665', '0.6933'] || Avg Loss:0.7524
Epoch:0 || ranker Index:40 || Loss:['0.7551', '0.8067', '0.6875', '0.6856', '0.6880', '0.7062', '0.6889', '0.7174', '0.6887', '0.7298'] || Avg Loss:0.7434
Epoch:0 || ranker Index:50 || Loss:['0.7200', '0.6871', '0.6564', '0.7099', '0.6715', '0.7328', '0.7807', '0.7146', '0.6359', '0.7195'] || Avg Loss:0.7354
Epoch:0 || ranker Index:60 || Loss:['0.7169', '0.7121', '0.7352', '0.8065', '0.7208', '0.6609', '0.7360', '0.7657', '0.6403', '0.6742'] || Avg Loss:0.7324
Epoch:0 || ranker Index:70 || Loss:['0.6997', '0.7249', '0.8

KeyboardInterrupt: 